# OmniVoice TTS - Commercial-Friendly TrainingTrain OmniVoice with **fresh Qwen3-0.6B** as the backbone and **randomly initialized audio components**.## Licensing (Commercial-Friendly)- **OmniVoice architecture (code):** Apache 2.0 ✅- **Qwen3-0.6B base model:** Apache 2.0 ✅- **Audio embeddings/heads:** Randomly initialized, trained on **your own data** ✅- **NOT using:** Pretrained OmniVoice weights (non-commercial) ❌Run on a free Tesla T4 Google Colab instance!

In [ ]:
%%captureimport os, reif "COLAB_" not in "".join(os.environ.keys()):    !pip install unslothelse:    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth    !unsloth install-kernels    !pip install --no-deps --upgrade "torchao>=0.16.0"!pip install transformers==5.15.0!pip install --no-deps trl==0.22.2!pip install --no-deps omnivoice==0.2.1import importlib.metadata as _mdtry: _tv = tuple(int(x) for x in _md.version("torch").split("+")[0].split(".")[:2])except Exception: _tv = (2, 11)_torchcodec = {(2, 11): "torchcodec>=0.11.0", (2, 10): "torchcodec==0.10.*", (2, 9): "torchcodec==0.9.*", (2, 8): "torchcodec==0.7.*", (2, 7): "torchcodec==0.5.*"}.get(_tv, "torchcodec>=0.12.0" if _tv >= (2, 12) else "torchcodec")!pip install pydub "audioop-lts; python_version >= '3.13'" webdataset soundfile librosa "{_torchcodec}" "datasets>=3.4.1,<4.0.0"import importlib.util, importlib.metadataif importlib.util.find_spec("torchaudio") is None:    _torch = importlib.metadata.version("torch")    _torchaudio = "torchaudio<=" + _torch.split("+")[0]    _index = "--index-url https://download.pytorch.org/whl/" + _torch.split("+")[1] if "+" in _torch else ""    !pip install "{_torchaudio}" {_index}

## UnslothOmniVoice is a `Qwen3-0.6B` backbone with audio embeddings and audio heads over 8 audio codebooks. It is a **diffusion language model**: attention is bidirectional and it fills in masked audio tokens over a few steps.We use **fresh Qwen3-0.6B** (Apache 2.0) as the backbone and initialize audio components **randomly** — no pretrained OmniVoice weights are used, making this fully commercial-friendly.

In [ ]:
from unsloth import FastModelimport torchfrom omnivoice import OmniVoicefrom transformers import AutoTokenizer, Qwen3ForCausalLMfrom torch.nn.init import normal_# Load OmniVoice architecture ONLY (to get structure + tokenizers)omnivoice_ref = OmniVoice.from_pretrained(    "k2-fsa/OmniVoice",    dtype = torch.float32,    train = True,)# Keep preprocessing components (not model weights)audio_tokenizer   = omnivoice_ref.audio_tokenizerfeature_extractor = omnivoice_ref.feature_extractortext_tokenizer    = omnivoice_ref.text_tokenizerconfig            = omnivoice_ref.configdel omnivoice_reftorch.cuda.empty_cache()# Load fresh Qwen3-0.6B backbone (Apache 2.0, commercial OK)backbone = Qwen3ForCausalLM.from_pretrained(    "Qwen/Qwen3-0.6B",    torch_dtype = torch.float32,    trust_remote_code = True,)# Build OmniVoice with fresh backbonedtype = torch.float32omnivoice = OmniVoice.from_pretrained(    "k2-fsa/OmniVoice",    dtype = dtype,    attn_implementation = "sdpa",    device_map = "cuda",)omnivoice.llm = backbone# Re-initialize audio_embeddings and audio_heads with RANDOM weights# ZERO pretrained (non-commercial) weights remainfor name, module in omnivoice.named_modules():    if "audio_embeddings" in name or "audio_heads" in name:        for param in module.parameters():            if param.requires_grad:                normal_(param, mean = 0.0, std = 0.02)omnivoice.audio_embeddings.float()omnivoice.audio_heads.float()print(f"Backbone: fresh Qwen3-0.6B")print(f"Audio components: randomly initialized")print(f"Config: {config}")

## LoRA vs Full FinetuningSet `FULL_FINETUNING = True` for full finetuning (all backbone params trainable, more VRAM).Set `FULL_FINETUNING = False` for LoRA (only ~1-10% of params trainable, much less VRAM).Audio embeddings and heads are **always** trained fully from scratch.

In [ ]:
FULL_FINETUNING = False  # <-- True = full finetune, False = LoRA# Save backbone so FastModel can load itbackbone.save_pretrained("qwen3_0.6b_backbone")AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B").save_pretrained("qwen3_0.6b_backbone")del backbonemodel, tokenizer = FastModel.from_pretrained(    model_name = "qwen3_0.6b_backbone",    max_seq_length = 2048,    dtype = None,            # auto: bfloat16 on Ampere+, float16 on T4    load_in_4bit = False,    full_finetuning = FULL_FINETUNING,    attn_implementation = "sdpa",)# Add LoRA (only if NOT full finetuning)if not FULL_FINETUNING:    model = FastModel.get_peft_model(        model,        r = 32,        target_modules = [            "q_proj", "k_proj", "v_proj", "o_proj",            "gate_proj", "up_proj", "down_proj",        ],        lora_alpha = 32,        lora_dropout = 0,        bias = "none",        use_gradient_checkpointing = "unsloth",        random_state = 3407,        use_rslora = False,        loftq_config = None,    )# Re-assemble: put trained backbone back into OmniVoiceomnivoice.llm = model.base_model.model.modelomnivoice.audio_embeddings.float()omnivoice.audio_heads.float()# Set requires_gradfor name, param in omnivoice.named_parameters():    if name.startswith(("audio_embeddings", "audio_heads")):        param.requires_grad = True    elif FULL_FINETUNING:        param.requires_grad = True    else:        param.requires_grad = "lora_" in namefor name, param in omnivoice.named_parameters():    if name.startswith("audio_tokenizer."):        param.requires_grad = Falsetorch.cuda.empty_cache()params = [p for n, p in omnivoice.named_parameters() if not n.startswith("audio_tokenizer.")]trainable = sum(p.numel() for p in params if p.requires_grad)total    = sum(p.numel() for p in params)print(f"Trainable parameters: {trainable:,} of {total:,} ({100*trainable/total:.2f}%)")print(f"Mode: {'FULL FINETUNING' if FULL_FINETUNING else 'LORA (r=32)'}")

## Data PrepWe use the `Etherll/kaira` dataset (text + audio). You can replace this with your own dataset.OmniVoice works on audio tokens from the Higgs audio tokenizer at 24kHz.

In [ ]:
from datasets import load_dataset, Audiodataset = load_dataset("Etherll/kaira", split = "train")dataset = dataset.cast_column("audio", Audio(sampling_rate = 24000))reference = dataset[0]print(f"Dataset size: {len(dataset)}")print(f"Sample: {dataset[0]}")

In [ ]:
# Tokenize audiodef tokenize_audio(example):    inputs = feature_extractor(        raw_audio = example["audio"]["array"],        sampling_rate = 24000,        return_tensors = "pt",    ).to(audio_tokenizer.device)    with torch.inference_mode():        audio_codes = audio_tokenizer.encode(inputs["input_values"]).audio_codes    return {"audio_tokens": audio_codes.squeeze(0).to(torch.int16).cpu().numpy()}dataset = dataset.map(tokenize_audio, remove_columns = ["audio"])

In [ ]:
from omnivoice.data.processor import OmniVoiceSampleProcessorfrom omnivoice.data.collator import PaddingDataCollatorprocessor = OmniVoiceSampleProcessor(    text_tokenizer = text_tokenizer,    num_channels = omnivoice.config.num_audio_codebook,    audio_mask_id = omnivoice.config.audio_mask_id,    prompt_ratio_range = (0.0, 0.3),    mask_ratio_range = (0.0, 1.0),    drop_cond_ratio = 0.1,    language_ratio = 0.8,    use_pinyin_ratio = 0.0,    instruct_ratio = 0.0,    only_instruct_ratio = 0.0,)class OmniVoiceDataset(torch.utils.data.Dataset):    def __init__(self, dataset, language = "en"):        self.dataset = dataset        self.language = language    def __len__(self):        return len(self.dataset)    def __getitem__(self, idx):        row = self.dataset[idx]        return processor({            "audio_tokens": torch.tensor(row["audio_tokens"]),            "label": {"text": row["text"], "language_id": self.language},        })train_dataset = OmniVoiceDataset(dataset)data_collator = PaddingDataCollator(processor, batch_tokens = None)

## TrainingTrain with HuggingFace `Trainer`. We do 60 steps here for speed; set `num_train_epochs=1` for a full run.

In [ ]:
from transformers import Trainer, TrainingArgumentsfrom unsloth import is_bfloat16_supportedtrainer = Trainer(    model = omnivoice,    train_dataset = train_dataset,    data_collator = data_collator,    args = TrainingArguments(        per_device_train_batch_size = 4,        gradient_accumulation_steps = 2,        warmup_steps = 5,        # num_train_epochs = 1,  # Uncomment for full run        max_steps = 60,        learning_rate = 2e-4,        fp16 = not is_bfloat16_supported(),        bf16 = is_bfloat16_supported(),        logging_steps = 1,        optim = "adamw_8bit",        weight_decay = 0.01,        max_grad_norm = 1.0,        lr_scheduler_type = "linear",        seed = 3407,        output_dir = "outputs",        remove_unused_columns = False,        report_to = "none",    ),)trainer_stats = trainer.train()print(f"Training complete. Runtime: {trainer_stats.metrics['train_runtime']:.1f} seconds")

## InferenceRun the model! OmniVoice can clone a voice from a short clip and its transcript.

In [ ]:
# Inferencefrom omnivoice import OmniVoice# The model is already loaded as `omnivoice`# You can use it directly for inferenceprint("Model ready for inference!")print(f"Audio tokenizer: {type(audio_tokenizer).__name__}")print(f"Feature extractor: {type(feature_extractor).__name__}")

## Save ModelSave the trained model for later use.

In [ ]:
# Save the trained OmniVoice modelomnivoice.save_pretrained("omnivoice_trained")print("Model saved to omnivoice_trained/")